# Pokemon data collection

Team Error_105 course project. See the repository README for inputs, working directories and scope.


In [ ]:
from pathlib import Path
for directory in ['../../data/pokemon_data/generation_pokemon', '../../data/pokemon_data/pokemon_details', '../../data/pokemon_data/species_details']:
    Path(directory).mkdir(parents=True, exist_ok=True)


In [ ]:
import json
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

In [ ]:
def get_generations_pokemon(generation: int):
    request_url = f'https://pokeapi.co/api/v2/generation/{generation}'
    response = requests.get(request_url, timeout=30)
    data = response.json()
    pokemon_df = pd.json_normalize(data, record_path = 'pokemon_species')
    pokemon_df['generation'] = generation
    return pokemon_df

In [ ]:
list_of_generations = [1, 2, 3, 4, 5, 6, 7, 8, 9]
list_of_df = [get_generations_pokemon(generation) for generation in list_of_generations]
poke_df = pd.concat(list_of_df)
display(poke_df)


In [ ]:
def extract_pokemon_id(url): 
    id = url.rstrip('/').split('/')[-1]
    return int(id)

poke_df['pokemon_id'] = poke_df['url'].apply(extract_pokemon_id)
poke_df = poke_df.sort_values('pokemon_id').reset_index(drop = True)
display(poke_df)


In [ ]:
poke_df.to_json('../../data/pokemon_data/generation_pokemon/generation_pokemon.json')

In [ ]:
def collect_pokemon_data(pokemon_id):
    try: 
        details_url = f'https://pokeapi.co/api/v2/pokemon/{pokemon_id}'
        details_response = requests.get(details_url, timeout=30)
        if details_response.status_code == 200:
            details_data = details_response.json()
            details_data['pokemon_portrait'] = details_data['sprites']['front_default']
            details_data.pop('sprites')
            with open(f'../../data/pokemon_data/pokemon_details/pokemon_{pokemon_id}_details.json', 'w') as file:
                json.dump(details_data, file)

        species_url = f'https://pokeapi.co/api/v2/pokemon-species/{pokemon_id}'
        species_response = requests.get(species_url, timeout=30)
        if species_response.status_code == 200:
            species_data = species_response.json()
            english_entry = next(entry for entry in species_data['flavor_text_entries'] if entry['language']['name'] == 'en')
            species_data['flavor_text_entries'] = english_entry
            with open(f'../../data/pokemon_data/species_details/pokemon_{pokemon_id}_species_details.json', 'w') as file:
                json.dump(species_data, file)       

        return 
    
    except Exception as e: 
        return f'Error for ID {pokemon_id}: {e}'

In [ ]:
def main(pokemon_ids): 
    with ThreadPoolExecutor() as executor: 
        results = list(executor.map(collect_pokemon_data, pokemon_ids))
    return results

In [ ]:
pokemon_ids = list(range(1,1026))
results = main(pokemon_ids)
print(f'Data successfully obtained for {results.count(None)} Pokemon.')